# CC_Dec25 — Data Cleaning, Attrition Audit & Time-Series EDA (v2)
**Forecasting granularity:** `Part Name` + `Region Id` (monthly) | **Horizon:** Oct 2022 – Dec 2025 (39 months)

**Changes vs. v1**
1. Retention % uses one consistent basis (positive quantity) in numerator and denominator.
2. Explicit date-window filter (own waterfall step) so no series can exceed 39 active months.
3. **Zero-filled monthly panel** (series x 39 months); all intermittency metrics are computed from it.
4. ADI/CV² follow Syntetos-Boylan: CV² on non-zero demand sizes (unrounded), ADI measured from first order to horizon end (trailing zeros count). The four quadrants are assigned (cutoffs 1.32 / 0.49).
5. Dormancy rule and its label now agree: no orders in Jul–Dec 2025 (last order >= 6 months before Dec 2025).
6. "Years active" for tiering counts only full years 2023–2025; 2022 is a 3-month stub.
7. Primary segment/site/bulk are volume-weighted (not transaction-count mode) and computed without slow lambdas.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
assert tuple(int(x) for x in pd.__version__.split(".")[:2]) >= (2, 0), "pandas >= 2.0 required (format='mixed')"

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

# ============================ CONFIGURATION ============================
FILE_PATH  = Path("CC_Dec25.csv")      # .csv or .xlsx
SHEET_NAME = "HDA"                     # used only for .xlsx
OUTPUT_DIR = Path("eda_outputs"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START = pd.Period("2022-10", freq="M")
END   = pd.Period("2025-12", freq="M")
ALL_PERIODS = pd.period_range(START, END, freq="M")
N_MONTHS = len(ALL_PERIODS)            # 39
DORMANT_MONTHS = 6                     # last order >= 6 months before END  => no orders Jul-Dec 2025
SBC_ADI_CUT, SBC_CV2_CUT = 1.32, 0.49  # Syntetos-Boylan cutoffs

## 1. Ingestion & raw schema audit

In [ ]:
STRING_COLS = ["Part Name", "Part Site", "Bulk Value", "Business Segment Value",
               "Business Segment Description", "Region Id"]

def load_raw_dataset(path: Path, sheet_name: str = "HDA") -> pd.DataFrame:
    dtype_map = {c: "string" for c in STRING_COLS}
    dtype_map["Quantity"] = "string"          # parsed manually (thousand separators)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path, sheet_name=sheet_name, dtype=dtype_map)
    else:
        df = pd.read_csv(path, dtype=dtype_map, low_memory=False, keep_default_na=False)  # keep_default_na=False: a Region Id of "NA" must not become NaN
    df.columns = [c.strip() for c in df.columns]

    blank_tokens = {"", "nan", "none", "null", "n/a", "-"}   # "NA" deliberately NOT treated as blank (could be a region code)
    for col in STRING_COLS:
        s = df[col].astype("string").str.strip()
        df[col] = s.mask(s.str.lower().isin(blank_tokens), pd.NA)

    q = df["Quantity"].astype("string").str.replace(",", "", regex=False).str.strip()
    df["Quantity"] = pd.to_numeric(q, errors="coerce")
    df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True, errors="coerce")
    return df

raw_df = load_raw_dataset(FILE_PATH, SHEET_NAME)
raw_df["Part_Region_Key"] = raw_df["Part Name"].fillna("MISSING") + " | " + raw_df["Region Id"].fillna("MISSING")

print(f"Raw shape               : {raw_df.shape[0]:,} rows x {raw_df.shape[1]-1} columns")
print(f"Raw date range          : {raw_df['Date'].min():%Y-%m-%d} to {raw_df['Date'].max():%Y-%m-%d}")
print(f"Unparseable dates       : {raw_df['Date'].isna().sum():,}")
print(f"Unparseable quantities  : {raw_df['Quantity'].isna().sum():,}")
print(f"Raw net quantity        : {raw_df['Quantity'].sum():,.2f}")
print(f"Raw positive quantity   : {raw_df.loc[raw_df['Quantity'] > 0, 'Quantity'].sum():,.2f}")

_p = raw_df.drop(columns=["Part_Region_Key"])
schema_audit = pd.DataFrame({
    "Dtype": _p.dtypes.astype(str), "Non_Null_Rows": _p.notna().sum(),
    "Blank_or_Null_Rows": _p.isna().sum(), "Blank_%": (_p.isna().mean() * 100).round(2),
    "Unique_Values": _p.nunique(dropna=True)})
display(Markdown("### Raw schema & missingness"))
display(schema_audit)

## 2. Cleaning rules & attrition analysis
Rules: (1) `Quantity <= 0` or NaN, (2) blank `Business Segment Description`, (3) `Part Name` starts with `9`, (4) blank `Bulk Value`,
plus safety steps: (5) missing `Part Name` / `Region Id` / unparseable `Date`, (6) date outside Oct 2022 – Dec 2025.
**Retention basis:** positive quantity for every row of the table (clean data contains only positive quantity, so the basis is consistent).

In [ ]:
d_period = raw_df["Date"].dt.to_period("M")

mask_qty    = raw_df["Quantity"].isna() | (raw_df["Quantity"] <= 0.0)
mask_seg    = raw_df["Business Segment Description"].isna()
mask_part9  = raw_df["Part Name"].fillna("").str.startswith("9")
mask_bulk   = raw_df["Bulk Value"].isna()
mask_key    = raw_df["Part Name"].isna() | raw_df["Region Id"].isna() | raw_df["Date"].isna()
mask_window = raw_df["Date"].notna() & ((d_period < START) | (d_period > END))

total_raw_rows    = len(raw_df)
total_raw_pos_qty = raw_df.loc[raw_df["Quantity"] > 0, "Quantity"].sum()

# ---- A. Independent rule impact ----
rules = {
    "Rule 1: Quantity <= 0 (or NaN)": mask_qty,
    "Rule 2: Business Segment Description blank": mask_seg,
    "Rule 3: Part Name starts with '9'": mask_part9,
    "Rule 4: Bulk Value blank": mask_bulk,
    "Safety: Part Name / Region Id / Date missing": mask_key,
    "Safety: Date outside Oct-22..Dec-25": mask_window,
}
rows = []
for name, m in rules.items():
    f = raw_df[m]
    pos = f.loc[f["Quantity"] > 0, "Quantity"].sum()
    rows.append({
        "Cleaning_Rule": name, "Flagged_Rows": len(f),
        "Flagged_Rows_%": round(len(f) / total_raw_rows * 100, 2),
        "Zero_Qty_Rows": int((f["Quantity"] == 0).sum()),
        "Negative_Qty_Rows": int((f["Quantity"] < 0).sum()),
        "Positive_Qty_Affected": pos,
        "Positive_Qty_%": round(pos / total_raw_pos_qty * 100, 2),
        "Negative_Qty_Affected": f.loc[f["Quantity"] < 0, "Quantity"].sum(),
        "Unique_Parts_Affected": f["Part Name"].nunique(dropna=True),
        "Unique_Part_Regions_Affected": f["Part_Region_Key"].nunique()})
indep_df = pd.DataFrame(rows)
display(Markdown("### A. Independent rule impact (overlaps not removed)"))
display(indep_df)

# ---- B. Sequential waterfall ----
steps = [
    ("0. Raw dataset", None),
    ("1. Remove Quantity <= 0", mask_qty),
    ("2. Remove blank Segment Description", mask_seg),
    ("3. Remove Part Name starting '9'", mask_part9),
    ("4. Remove blank Bulk Value", mask_bulk),
    ("5. Remove missing Part/Region/Date", mask_key),
    ("6. Remove dates outside window", mask_window),
]
curr = raw_df.copy()
wf = []
for name, m in steps:
    if m is None:
        removed, removed_pos = 0, 0.0
    else:
        mm = m.loc[curr.index]
        dropped = curr[mm]
        removed = len(dropped)
        removed_pos = dropped.loc[dropped["Quantity"] > 0, "Quantity"].sum()
        curr = curr[~mm]
    rem_pos = curr.loc[curr["Quantity"] > 0, "Quantity"].sum()
    wf.append({"Step": name, "Rows_Removed": removed, "Rows_Remaining": len(curr),
               "Retained_Rows_%": round(len(curr) / total_raw_rows * 100, 2),
               "Pos_Qty_Removed": removed_pos, "Pos_Qty_Remaining": rem_pos,
               "Retained_Pos_Qty_%": round(rem_pos / total_raw_pos_qty * 100, 2),
               "Unique_Parts": curr["Part Name"].nunique(), "Unique_Regions": curr["Region Id"].nunique(),
               "Unique_Part_Region_Pairs": curr["Part_Region_Key"].nunique()})
waterfall_df = pd.DataFrame(wf)
display(Markdown("### B. Sequential cleaning waterfall"))
display(waterfall_df)

# ---- C. By region ----
def _cnt(m): return raw_df[m].groupby("Region Id", dropna=False).size()
diag_by_region = pd.DataFrame({
    "Raw_Rows": raw_df.groupby("Region Id", dropna=False).size(),
    "Flag_Qty<=0": _cnt(mask_qty), "Flag_Blank_Seg": _cnt(mask_seg),
    "Flag_Part_9xx": _cnt(mask_part9), "Flag_Blank_Bulk": _cnt(mask_bulk),
    "Clean_Rows": curr.groupby("Region Id", dropna=False).size()}).fillna(0).astype(int)
diag_by_region["Retention_%"] = (diag_by_region["Clean_Rows"] / diag_by_region["Raw_Rows"] * 100).round(2)
display(Markdown("### C. Attrition by Region Id (flags are independent, not additive)"))
display(diag_by_region.sort_values("Raw_Rows", ascending=False))

# ---- Plot ----
labels = ["Raw", "Qty>0", "Seg", "No '9'", "Bulk", "Keys", "Window"]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, col, pct, color, title in [
    (axes[0], "Rows_Remaining", "Retained_Rows_%", "#2b5c8f", "Row retention"),
    (axes[1], "Pos_Qty_Remaining", "Retained_Pos_Qty_%", "#2e8b57", "Positive-quantity retention")]:
    ax.bar(labels, waterfall_df[col], color=color, alpha=0.85)
    ax.set_title(title); ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
    for i, (v, p) in enumerate(zip(waterfall_df[col], waterfall_df[pct])):
        ax.text(i, v, f"{v:,.0f}\n({p}%)", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

clean_df = curr.copy()
assert (clean_df["Quantity"] > 0).all()
clean_df["Year"] = clean_df["Date"].dt.year
clean_df["Month"] = clean_df["Date"].dt.month
clean_df["YearMonth"] = clean_df["Date"].dt.to_period("M")
clean_df["Ord"] = clean_df["Year"] * 12 + clean_df["Month"] - 1     # integer month index

## 3. Core EDA & granularity integrity check

In [ ]:
display(Markdown("### A. Clean dataset summary"))
display(pd.DataFrame({
    "Metric": ["Clean transaction rows", "Date range", "Calendar months covered", "Total clean quantity",
               "Unique Part Names", "Unique Region Ids", "Unique (Part Name + Region Id) series",
               "Unique Part Sites", "Unique Bulk Values", "Unique Segment Values", "Unique Segment Descriptions"],
    "Value": [f"{len(clean_df):,}", f"{clean_df['Date'].min():%Y-%m-%d} to {clean_df['Date'].max():%Y-%m-%d}",
              f"{clean_df['YearMonth'].nunique()}", f"{clean_df['Quantity'].sum():,.2f}",
              f"{clean_df['Part Name'].nunique():,}", f"{clean_df['Region Id'].nunique():,}",
              f"{clean_df['Part_Region_Key'].nunique():,}", f"{clean_df['Part Site'].nunique():,}",
              f"{clean_df['Bulk Value'].nunique():,}", f"{clean_df['Business Segment Value'].nunique():,}",
              f"{clean_df['Business Segment Description'].nunique():,}"]}))

# ---- B. Granularity integrity ----
attr_card = clean_df.groupby(["Part Name", "Region Id"], as_index=False).agg(
    Distinct_Sites=("Part Site", "nunique"), Distinct_Bulk=("Bulk Value", "nunique"),
    Distinct_Seg_Values=("Business Segment Value", "nunique"),
    Distinct_Seg_Descs=("Business Segment Description", "nunique"),
    Transactions=("Quantity", "count"), Total_Quantity=("Quantity", "sum"))
cols = {"Part Site": "Distinct_Sites", "Bulk Value": "Distinct_Bulk",
        "Business Segment Value": "Distinct_Seg_Values", "Business Segment Description": "Distinct_Seg_Descs"}
multi = pd.DataFrame({"Attribute": list(cols),
    "Pairs_With_1_Value": [(attr_card[c] == 1).sum() for c in cols.values()],
    "Pairs_With_Multiple_Values": [(attr_card[c] > 1).sum() for c in cols.values()]})
multi["Multi_Value_%"] = (multi["Pairs_With_Multiple_Values"] / len(attr_card) * 100).round(2)
multi["Volume_In_Multi_Value_Pairs_%"] = [
    round(attr_card.loc[attr_card[c] > 1, "Total_Quantity"].sum() / attr_card["Total_Quantity"].sum() * 100, 2)
    for c in cols.values()]
display(Markdown("### B. Attribute cardinality per `Part Name + Region Id`"))
display(multi)

# ---- C. Monthly aggregation (sparse: only months with orders) ----
monthly_df = (clean_df.groupby(["Part Name", "Region Id", "Year", "Month", "Ord"], as_index=False)
              .agg(Monthly_Qty=("Quantity", "sum"), Order_Count=("Quantity", "count")))
monthly_df["YearMonth"] = pd.PeriodIndex.from_fields(
    year=monthly_df["Year"], month=monthly_df["Month"], freq="M") if hasattr(pd.PeriodIndex, "from_fields") else pd.PeriodIndex(
    year=monthly_df["Year"], month=monthly_df["Month"], freq="M")

# ---- D. Volume-weighted primary attributes per series ----
def primary_by_volume(df, col, new):
    g = df.groupby(["Part Name", "Region Id", col], as_index=False)["Quantity"].sum()
    g = g.sort_values(["Part Name", "Region Id", "Quantity"], ascending=[True, True, False])
    return g.drop_duplicates(["Part Name", "Region Id"])[["Part Name", "Region Id", col]].rename(columns={col: new})

primary_attrs = primary_by_volume(clean_df, "Business Segment Description", "Primary_Segment")
for c, n in [("Part Site", "Primary_Site"), ("Bulk Value", "Primary_Bulk")]:
    primary_attrs = primary_attrs.merge(primary_by_volume(clean_df, c, n), on=["Part Name", "Region Id"])
primary_attrs = primary_attrs.merge(attr_card[["Part Name", "Region Id", "Distinct_Seg_Descs"]]
                                    .rename(columns={"Distinct_Seg_Descs": "N_Segments"}), on=["Part Name", "Region Id"])

display(Markdown("### C. Quantity distribution: transactional vs monthly-aggregated (non-zero months only)"))
q = [0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
display(pd.DataFrame({"Transactional_Quantity": clean_df["Quantity"].describe(percentiles=q),
                      "Monthly_Part_Region_Qty": monthly_df["Monthly_Qty"].describe(percentiles=q)}))

## 4. Zero-filled monthly panel, demand profile, year cohorts & forecastability
`panel` has one row per `Part Name + Region Id` and exactly 39 monthly columns (Oct 2022 – Dec 2025); months without orders are **0**.

In [ ]:
# ---------------- Zero-filled panel ----------------
ord_idx = [p.ordinal for p in ALL_PERIODS]                    # absolute month ordinals
def _to_ord(y, m): return y * 12 + m - 1
panel = (monthly_df.pivot_table(index=["Part Name", "Region Id"], columns="Ord",
                                values="Monthly_Qty", aggfunc="sum")
         .reindex(columns=[_to_ord(p.year, p.month) for p in ALL_PERIODS]).fillna(0.0))
panel.columns = [str(p) for p in ALL_PERIODS]
assert panel.shape[1] == N_MONTHS
assert np.isclose(panel.to_numpy().sum(), clean_df["Quantity"].sum()), "panel volume != clean volume"
print(f"Panel: {panel.shape[0]:,} series x {panel.shape[1]} months | zero cells: {(panel.to_numpy()==0).mean()*100:.1f}%")

vals = panel.to_numpy(); nz = vals > 0
col_years = np.array([p.year for p in ALL_PERIODS])

sp = pd.DataFrame(index=panel.index).reset_index()
sp["Part_Region_Key"] = sp["Part Name"] + " | " + sp["Region Id"]
sp["Active_Months"]   = nz.sum(1)
first_i = nz.argmax(1); last_i = N_MONTHS - 1 - nz[:, ::-1].argmax(1)
sp["First_Month"] = [str(ALL_PERIODS[i]) for i in first_i]
sp["Last_Month"]  = [str(ALL_PERIODS[i]) for i in last_i]
sp["Total_Quantity"] = vals.sum(1)
sp["Span_To_Last_Months"] = last_i - first_i + 1
sp["Span_To_End_Months"]  = N_MONTHS - first_i                 # first order -> Dec 2025 (trailing zeros count)
sp["Months_Since_Last_Order"] = (N_MONTHS - 1) - last_i
sp["Fill_Rate_To_End_%"] = (sp["Active_Months"] / sp["Span_To_End_Months"] * 100).round(1)

for yr in [2022, 2023, 2024, 2025]:
    sp[f"Mos_{yr}"] = nz[:, col_years == yr].sum(1)
sp["Num_Full_Years_Active"] = sum((sp[f"Mos_{y}"] > 0).astype(int) for y in [2023, 2024, 2025])  # 2022 is a 3-month stub

def year_combo(r):
    ys = [str(y) for y in [2022, 2023, 2024, 2025] if r[f"Mos_{y}"] > 0]
    return f"{ys[0]} Only" if len(ys) == 1 else " - ".join(ys)
sp["Year_Iteration"] = sp.apply(year_combo, axis=1)

# ---- Syntetos-Boylan: ADI to horizon end; CV^2 on non-zero demand sizes (unrounded) ----
nzv = panel.where(panel > 0)
mean_nz = nzv.mean(axis=1).to_numpy(); std_nz = nzv.std(axis=1, ddof=1).to_numpy()
sp["Mean_NonZero_Qty"] = mean_nz
sp["Std_NonZero_Qty"] = std_nz
sp["ADI"] = sp["Span_To_End_Months"] / sp["Active_Months"]
sp["CV2"] = (std_nz / mean_nz) ** 2                              # NaN when only 1 non-zero month
sp["SBC_Class"] = np.select(
    [sp["CV2"].isna(),
     (sp["ADI"] < SBC_ADI_CUT) & (sp["CV2"] < SBC_CV2_CUT),
     (sp["ADI"] < SBC_ADI_CUT) & (sp["CV2"] >= SBC_CV2_CUT),
     (sp["ADI"] >= SBC_ADI_CUT) & (sp["CV2"] < SBC_CV2_CUT)],
    ["Insufficient (1 non-zero month)", "Smooth", "Erratic", "Intermittent"], default="Lumpy")

# orders & attributes
ords = monthly_df.groupby(["Part Name", "Region Id"], as_index=False)["Order_Count"].sum().rename(columns={"Order_Count": "Total_Orders"})
sp = sp.merge(ords, on=["Part Name", "Region Id"], how="left").merge(primary_attrs, on=["Part Name", "Region Id"], how="left")

# ---- Forecastability triage (label and rule now agree) ----
def triage(r):
    if r["Months_Since_Last_Order"] >= DORMANT_MONTHS:
        return "4. Dormant (no orders Jul-Dec 2025)"
    if r["Year_Iteration"] == "2025 Only":
        return "3. Cold-start (first order in 2025; needs analog/pooling)"
    if r["Active_Months"] >= 18 and r["Num_Full_Years_Active"] == 3:
        return "1. Tier-1 (orders in 2023, 2024 & 2025; >=18 active months)"
    if r["Active_Months"] >= 10 and r["Num_Full_Years_Active"] >= 2:
        return "2. Tier-2 (>=2 of 2023-25; 10-17 active months)"
    return "5. Ultra-sparse / intermittent"
sp["Forecastability_Tier"] = sp.apply(triage, axis=1)
series_profile = sp
total_series, total_vol = len(sp), sp["Total_Quantity"].sum()

# ---------------- A. Year-presence iterations ----------------
yc = (sp.groupby("Year_Iteration", as_index=False)
      .agg(Part_Region_Count=("Part_Region_Key", "count"), Unique_Parts=("Part Name", "nunique"),
           Total_Quantity=("Total_Quantity", "sum"), Avg_Active_Months=("Active_Months", "mean"),
           Median_Active_Months=("Active_Months", "median"), Min_Active_Months=("Active_Months", "min"),
           Max_Active_Months=("Active_Months", "max"),
           Active_Last_6M=("Months_Since_Last_Order", lambda s: (s < DORMANT_MONTHS).sum()),
           Active_In_Q4_2025=("Months_Since_Last_Order", lambda s: (s <= 2).sum()))
      .sort_values("Part_Region_Count", ascending=False))
yc["Series_Share_%"] = (yc["Part_Region_Count"] / total_series * 100).round(2)
yc["Volume_Share_%"] = (yc["Total_Quantity"] / total_vol * 100).round(2)
yc["Avg_Active_Months"] = yc["Avg_Active_Months"].round(1)
year_cohort_report = yc[["Year_Iteration", "Part_Region_Count", "Series_Share_%", "Unique_Parts", "Total_Quantity",
                         "Volume_Share_%", "Avg_Active_Months", "Median_Active_Months", "Min_Active_Months",
                         "Max_Active_Months", "Active_Last_6M", "Active_In_Q4_2025"]]
display(Markdown("### A. Year-presence iterations (2022 = Oct-Dec stub only)"))
display(year_cohort_report)

# ---------------- B. Active-month buckets ----------------
assert sp["Active_Months"].between(1, N_MONTHS).all()
bins = [0, 1, 3, 6, 12, 18, 24, 30, 36, N_MONTHS]
blabels = ["01 mo", "02-03 mos", "04-06 mos", "07-12 mos", "13-18 mos", "19-24 mos", "25-30 mos", "31-36 mos", "37-39 mos"]
sp["Months_Bucket"] = pd.cut(sp["Active_Months"], bins=bins, labels=blabels)
months_bucket_report = (sp.groupby("Months_Bucket", observed=False, as_index=False)
    .agg(Part_Region_Count=("Part_Region_Key", "count"), Total_Quantity=("Total_Quantity", "sum"),
         Active_In_2025=("Mos_2025", lambda s: (s > 0).sum()),
         Active_Last_6M=("Months_Since_Last_Order", lambda s: (s < DORMANT_MONTHS).sum())))
months_bucket_report["Series_Share_%"] = (months_bucket_report["Part_Region_Count"] / total_series * 100).round(2)
months_bucket_report["Volume_Share_%"] = (months_bucket_report["Total_Quantity"] / total_vol * 100).round(2)
assert months_bucket_report["Part_Region_Count"].sum() == total_series
display(Markdown("### B. Active-months distribution"))
display(months_bucket_report)

# ---------------- C. Triage ----------------
triage_report = (sp.groupby("Forecastability_Tier", as_index=False)
    .agg(Part_Region_Count=("Part_Region_Key", "count"), Unique_Parts=("Part Name", "nunique"),
         Total_Quantity=("Total_Quantity", "sum"), Avg_Active_Months=("Active_Months", "mean")))
triage_report["Series_Share_%"] = (triage_report["Part_Region_Count"] / total_series * 100).round(2)
triage_report["Volume_Share_%"] = (triage_report["Total_Quantity"] / total_vol * 100).round(2)
triage_report["Avg_Active_Months"] = triage_report["Avg_Active_Months"].round(1)
display(Markdown("### C. Forecasting-readiness triage"))
display(triage_report)

# ---------------- D. Syntetos-Boylan classification ----------------
sbc_report = (sp.groupby("SBC_Class", as_index=False)
    .agg(Part_Region_Count=("Part_Region_Key", "count"), Total_Quantity=("Total_Quantity", "sum"),
         Median_ADI=("ADI", "median"), Median_CV2=("CV2", "median")))
sbc_report["Series_Share_%"] = (sbc_report["Part_Region_Count"] / total_series * 100).round(2)
sbc_report["Volume_Share_%"] = (sbc_report["Total_Quantity"] / total_vol * 100).round(2)
display(Markdown("### D. Demand pattern (SBC: ADI cut 1.32, CV² cut 0.49; ADI incl. trailing zeros)"))
display(sbc_report.sort_values("Part_Region_Count", ascending=False))
display(Markdown("**Tier x SBC class (series count)**"))
display(pd.crosstab(sp["Forecastability_Tier"], sp["SBC_Class"], margins=True))

## 5. Regional, cross-region & business-segment EDA

In [ ]:
sp["Is_T1_T2"] = sp["Forecastability_Tier"].str.startswith(("1.", "2."))
sp["T12_Qty"] = sp["Total_Quantity"].where(sp["Is_T1_T2"], 0.0)

region_summary = (sp.groupby("Region Id", as_index=False)
    .agg(Unique_Part_Names=("Part Name", "nunique"), Part_Region_Series=("Part_Region_Key", "count"),
         Total_Quantity=("Total_Quantity", "sum"), Total_Orders=("Total_Orders", "sum"),
         Avg_Qty_Per_Part=("Total_Quantity", "mean"), Median_Qty_Per_Part=("Total_Quantity", "median"),
         Avg_Active_Months=("Active_Months", "mean"), Tier1_Tier2_Series=("Is_T1_T2", "sum"),
         Tier1_Tier2_Volume=("T12_Qty", "sum"))
    .sort_values("Total_Quantity", ascending=False))
region_summary["Volume_Share_%"] = (region_summary["Total_Quantity"] / total_vol * 100).round(2)
region_summary["Series_Share_%"] = (region_summary["Part_Region_Series"] / total_series * 100).round(2)
region_summary["Forecastable_Series_%"] = (region_summary["Tier1_Tier2_Series"] / region_summary["Part_Region_Series"] * 100).round(1)
region_summary["Forecastable_Vol_%"] = (region_summary["Tier1_Tier2_Volume"] / region_summary["Total_Quantity"] * 100).round(1)
region_summary["Avg_Active_Months"] = region_summary["Avg_Active_Months"].round(1)
display(Markdown("### A. Regional summary"))
display(region_summary)

part_footprint = (sp.groupby("Part Name", as_index=False)
    .agg(Region_Count=("Region Id", "nunique"), Regions_List=("Region Id", lambda s: ", ".join(sorted(s.unique()))),
         Total_Part_Qty=("Total_Quantity", "sum")))
footprint_dist = (part_footprint.groupby("Region_Count", as_index=False)
    .agg(Unique_Parts=("Part Name", "count"), Total_Quantity=("Total_Part_Qty", "sum")))
footprint_dist["Parts_Share_%"] = (footprint_dist["Unique_Parts"] / len(part_footprint) * 100).round(2)
footprint_dist["Volume_Share_%"] = (footprint_dist["Total_Quantity"] / total_vol * 100).round(2)
display(Markdown("### B. Cross-region part sharing"))
display(footprint_dist)

segment_summary = (clean_df.groupby(["Business Segment Value", "Business Segment Description"], as_index=False)
    .agg(Transactions=("Quantity", "count"), Unique_Parts=("Part Name", "nunique"),
         Unique_Part_Regions=("Part_Region_Key", "nunique"), Unique_Regions=("Region Id", "nunique"),
         Total_Quantity=("Quantity", "sum")).sort_values("Total_Quantity", ascending=False))
segment_summary["Volume_Share_%"] = (segment_summary["Total_Quantity"] / total_vol * 100).round(2)
display(Markdown("### C. Business segment breakdown"))
display(segment_summary)

display(Markdown("### D. Volume matrix: Region Id x Business Segment Description"))
display(pd.pivot_table(clean_df, index="Region Id", columns="Business Segment Description",
                       values="Quantity", aggfunc="sum", fill_value=0))

## 6. Time-series trends, seasonality & diagnostics

In [ ]:
tot = panel.sum(axis=0); active = (panel > 0).sum(axis=0)

# Fig 1: volume + active series
fig, ax1 = plt.subplots(figsize=(16, 5))
ax1.bar(panel.columns, tot.values, color="#2b5c8f", alpha=0.8)
ax1.set_ylabel("Total monthly quantity", color="#2b5c8f", fontweight="bold")
ax1.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}")); ax1.tick_params(axis="x", rotation=60)
ax2 = ax1.twinx()
ax2.plot(panel.columns, active.values, color="#d95f02", marker="o", linewidth=2)
ax2.set_ylabel("Active (Part + Region) series", color="#d95f02", fontweight="bold"); ax2.grid(False)
plt.title("Monthly demand volume & active series (Oct 2022 - Dec 2025)")
fig.tight_layout(); plt.show()

# Fig 2: year-presence cohorts
top = year_cohort_report.head(12).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(top["Year_Iteration"], top["Part_Region_Count"], color="#3b7ea1")
axes[0].set_title("Series count by year-presence iteration")
for i, (c, p) in enumerate(zip(top["Part_Region_Count"], top["Series_Share_%"])):
    axes[0].text(c, i, f" {c:,} ({p}%)", va="center", fontsize=9)
axes[1].barh(top["Year_Iteration"], top["Total_Quantity"], color="#b5457a")
axes[1].set_title("Volume by year-presence iteration")
axes[1].xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
for i, (q_, p) in enumerate(zip(top["Total_Quantity"], top["Volume_Share_%"])):
    axes[1].text(q_, i, f" {p}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()

# Fig 3: regional trend + seasonality heatmap
reg_m = panel.groupby(level="Region Id").sum().T
reg_m.index = pd.PeriodIndex(reg_m.index, freq="M").to_timestamp()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
reg_m.plot(ax=axes[0], marker="o", markersize=3)
axes[0].set_title("Monthly demand by Region Id"); axes[0].set_ylabel("Quantity")
axes[0].yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
season = monthly_df.pivot_table(index="Year", columns="Month", values="Monthly_Qty", aggfunc="sum").astype(float)
sns.heatmap(season / 1e6, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1], cbar_kws={"label": "Quantity (M)"})
axes[1].set_title("Seasonality: volume (millions) by year & month (2022 = Oct-Dec only)")
plt.tight_layout(); plt.show()

# Fig 4: active-months histogram
plt.figure(figsize=(14, 4))
plt.hist(sp["Active_Months"], bins=np.arange(0.5, N_MONTHS + 1.5, 1), color="#3b7ea1")
plt.axvline(12, color="orange", ls="--", label="12 months"); plt.axvline(24, color="green", ls="--", label="24 months")
plt.title(f"Active months per series (max = {N_MONTHS})"); plt.xlabel("Active months"); plt.ylabel("Series"); plt.legend()
plt.tight_layout(); plt.show()

# Fig 5: SBC scatter
d = sp.dropna(subset=["CV2"])
plt.figure(figsize=(9, 6))
for cls, g in d.groupby("SBC_Class"):
    plt.scatter(g["ADI"], g["CV2"].clip(lower=1e-3), s=10, alpha=0.5, label=f"{cls} ({len(g):,})")
plt.axvline(SBC_ADI_CUT, color="k", ls="--"); plt.axhline(SBC_CV2_CUT, color="k", ls="--")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("ADI (log)"); plt.ylabel("CV² of non-zero sizes (log)")
plt.title("Syntetos-Boylan demand classification"); plt.legend(markerscale=2)
plt.tight_layout(); plt.show()

## 7. Export

In [ ]:
clean_df.drop(columns=["Part_Region_Key", "YearMonth", "Ord"]).to_csv(OUTPUT_DIR / "clean_transactions.csv", index=False)
panel.reset_index().to_csv(OUTPUT_DIR / "monthly_panel_zero_filled_wide.csv", index=False)
(panel.reset_index().melt(id_vars=["Part Name", "Region Id"], var_name="YearMonth", value_name="Quantity")
      .to_csv(OUTPUT_DIR / "monthly_panel_zero_filled_long.csv", index=False))

series_export = sp.drop(columns=["Is_T1_T2", "T12_Qty"]).copy()
series_export["Months_Bucket"] = series_export["Months_Bucket"].astype(str)

with pd.ExcelWriter(OUTPUT_DIR / "CC_Dec25_EDA_Report.xlsx", engine="openpyxl") as w:
    waterfall_df.to_excel(w, sheet_name="1_Cleaning_Waterfall", index=False)
    indep_df.to_excel(w, sheet_name="2_Independent_Rule_Flags", index=False)
    year_cohort_report.to_excel(w, sheet_name="3_Year_Iterations", index=False)
    months_bucket_report.to_excel(w, sheet_name="4_Active_Months_Buckets", index=False)
    triage_report.to_excel(w, sheet_name="5_Forecastability_Triage", index=False)
    sbc_report.to_excel(w, sheet_name="6_SBC_Classes", index=False)
    region_summary.to_excel(w, sheet_name="7_Regional_EDA", index=False)
    segment_summary.to_excel(w, sheet_name="8_Segment_EDA", index=False)
    series_export.to_excel(w, sheet_name="9_Series_Master_List", index=False)
print(f"Saved to: {OUTPUT_DIR.resolve()}")